# 12/18
first try to compare to around WVR data to Balloon data 

In [ ]:
"""
read from floder ./Data/*.csv to have all am data (time and pwv)
read balloon data from ./Data/Alldate_balloon_SP.csv
regenerate a new csv by following conditions:
for each balloon data time, find am data within +/- 60 min
save new csv to ./around_balloon_am_data.csv
"""

import pandas as pd 
import numpy as np  
#import matplotlib.pyplot as plt
import glob
import os
from datetime import datetime, timedelta
import pickle
import matplotlib.pyplot as plt   

am_fn_list = glob.glob('./Data/time*.csv')
balloon_fn = './Data/Alldate_balloon_SP.csv'

am_fn_list.sort()
all_am_df = pd.DataFrame()

# with open('./around_balloon_am_data.csv', 'w+') as f:
#     f.write('Time(%Y-%m-%d %H:%M:%S), PWV(um) \n')

for am_fn in am_fn_list:
    am_df = pd.read_csv(am_fn, dtype={'DATE(YYYYMMDD)': str, 'START TIME(HHMMSS.Microsec)': str, 'PWV TOTAL zenith': str})
    am_df['TIME'] = pd.to_datetime(am_df['DATE(YYYYMMDD)'] + ' ' + am_df['START TIME(HHMMSS.Microsec)'], format='%Y%m%d %H%M%S.%f')
    am_df = am_df.sort_values(by='TIME') 
    am_TIME = am_df['TIME'].dt.strftime('%Y-%m-%d %H:%M:%S')
    am_PWV = am_df['PWV TOTAL zenith'].str.split(' ').str[0].astype(float)
    am_df_s = pd.DataFrame({'Time': am_TIME, 'PWV': am_PWV})
    all_am_df = pd.concat([all_am_df, am_df_s], ignore_index=True)

#print(f'Total AM data points: {len(all_am_df)} \n {all_am_df.head(3)} \n')

balloon_df = pd.read_csv(balloon_fn)
balloon_df['Date(YYYYMMDD_HHMMSS)'] = pd.to_datetime(balloon_df['Date(YYYYMMDD_HHMMSS)'], format='%Y%m%d_%H%M%S')
#print(f'Total Balloon data points: {len(balloon_df)} \n {balloon_df.head(3)} \n')

am_num_out = 0

for i in range(3, len(balloon_df)):
    num_balloon = i
    balloon_time = balloon_df.loc[num_balloon, 'Date(YYYYMMDD_HHMMSS)']
    balloon_time_dt = datetime.strptime(str(balloon_time), '%Y-%m-%d %H:%M:%S')
    balloon_date_fn = str(balloon_time_dt.strftime('%y%m%d_%H'))
    #print(f'Processing balloon time: {balloon_date_fn}')
    balloon_pwv = balloon_df.loc[num_balloon, 'PWV(um)']

    recorded_j = 0

    for j in range(len(all_am_df)):
        j = j + am_num_out
        if j >= len(all_am_df):
            print('Error')
            break
        num_am = j
        am_time = all_am_df.loc[num_am, 'Time']
        am_time_dt = datetime.strptime(am_time, '%Y-%m-%d %H:%M:%S')
        am_pwv = all_am_df.loc[num_am, 'PWV']

        if am_time_dt.day == balloon_time_dt.day  and am_time_dt.month == balloon_time_dt.month and am_time_dt.year == balloon_time_dt.year: 
            
            if abs((am_time_dt - balloon_time_dt).total_seconds()) <= 7200:
                if os.path.exists(f'around_{balloon_date_fn}.csv') is False:
                    with open(f'around_{balloon_date_fn}.csv', 'w+') as f:
                        f.write('TIME, am_PWV(um), balloon_PWV(um) \n')
                        f.write(f'{am_time}, {am_pwv}, {balloon_pwv} \n')
                else:
                    with open(f'around_{balloon_date_fn}.csv', 'a+') as f:
                        f.write(f'{am_time}, {am_pwv}, {balloon_pwv} \n')
            else:
                recorded_j = j
                break
        
        
    print(f"num {i} as {recorded_j} of {len(all_am_df)}")
    
    am_num_out = recorded_j

    #print(f'Done for balloon time: {balloon_date_fn} \n')





num 3 as 11 of 74165
Done for balloon time: 240103_00 

num 4 as 11 of 74165
Done for balloon time: 240103_12 

num 5 as 198 of 74165
Done for balloon time: 240104_00 

num 6 as 198 of 74165
Done for balloon time: 240104_12 

num 7 as 385 of 74165
Done for balloon time: 240105_00 

num 8 as 385 of 74165
Done for balloon time: 240105_12 

num 9 as 572 of 74165
Done for balloon time: 240106_00 

num 10 as 572 of 74165
Done for balloon time: 240106_12 

num 11 as 759 of 74165
Done for balloon time: 240107_00 

num 12 as 759 of 74165
Done for balloon time: 240107_12 

num 13 as 946 of 74165
Done for balloon time: 240108_00 

num 14 as 946 of 74165
Done for balloon time: 240108_12 

num 15 as 1133 of 74165
Done for balloon time: 240109_00 

num 16 as 1133 of 74165
Done for balloon time: 240109_12 

num 17 as 1320 of 74165
Done for balloon time: 240110_00 

num 18 as 1320 of 74165
Done for balloon time: 240110_12 

num 19 as 1507 of 74165
Done for balloon time: 240111_00 

num 20 as 1694 of 

KeyboardInterrupt: 